# Delft3D Binary (.dat/.def) to Sandplover DataCube Conversion

This notebook converts a **Delft3D** hydrodynamic model output in **NEFIS binary format** (`.dat` / `.def` file pair) into a **[Sandplover](https://github.com/sandpiper-toolchain/sandplover) DataCube** — a structured 3D array with dimensions `(time, x, y)`.

## What this does
The converter reads a Delft3D `trim-*.dat` / `trim-*.def` file pair, extracts and computes the following variables, and saves them as a Sandplover-compatible NetCDF DataCube:

| Output Variable | Source | Description |
|----------------|--------|-------------|
| `eta` | `−DPS` (map-sed-series) | Bed elevation |
| `water_depth` | `S1 − eta` | Water column thickness |
| `velocity` | `sqrt(U1² + V1²)` surface layer | Velocity magnitude |
| `mud_frac` | `MUDFRAC` | Mud fraction (0–1) |
| `sand_frac` | `1 − MUDFRAC` | Sand fraction (0–1) |

**Time axis** is taken from `MORFT` (morphological time in days, from `map-infsed-serie`).

## Prerequisites
- `sandplover` must be installed: `pip install sandplover`
- `xarray`, `numpy`, `pandas` must be installed
- `delft3d_dat_converter.py` must be in the same directory as this notebook

## Notes
- The NEFIS binary format is read with a pure-Python parser — no external NEFIS library is required.
- **x and y dimensions are swapped** in the output relative to the Delft3D grid convention (NMAX↔MMAX), consistent with `delft3d_converter.py`.
- To add more variables, see **Section 3** below.

## 1. Import Converter

Import `Delft3DDatConverter` from `delft3d_dat_converter.py`.  
This class handles the full pipeline:  
open .dat/.def → read NEFIS binary → extract variables → build Sandplover DataCube → save NetCDF.

In [11]:
from delft3d_dat_converter import Delft3DDatConverter

## 2. Run Conversion

Set `dat_file` to the path of your Delft3D `.dat` file (the `.def` file must be in the same folder with the same base name).  
Set `output_file` to the desired path for the resulting DataCube.

**Parameters for `.convert()`:**
- `output_file` — path where the output DataCube NetCDF will be saved
- `show_stats=True` — prints shape, value range, mean, and std for each variable after conversion

In [12]:
# Set file paths — change to your .dat file location
# The matching .def file must be in the same directory with the same base name.
dat_file    = 'trim-DV_run.dat'
output_file = 'sandplover_cube_dat.nc'

# Run conversion
converter = Delft3DDatConverter(dat_file).convert(output_file, show_stats=True)

Reading: trim-DV_run.dat
         trim-DV_run.def
Groups found: ['map-const', 'map-info-series', 'map-series', 'map-infsed-serie', 'map-sed-series', 'map-infavg-serie', 'map-avg-series', 'map-version', 'TEMPOUT']
Grid: NMAX=227, MMAX=302
Time steps: 37, MORFT range: [1245.8333, 1270.8333] days
eta (= -DPS): [-5.7322, 5.0000] m
water_depth: [0.0000, 7.3938] m
velocity:    [0.0000, 1.4721] m/s
Sediment fractions calculated
Variables: ['eta', 'water_depth', 'velocity', 'mud_frac', 'sand_frac']
Dimensions: time=37, x=302, y=227
DataCube created: (37, 302, 227) (time, x, y)
   Variables: ['eta', 'water_depth', 'velocity', 'mud_frac', 'sand_frac']
Saved: sandplover_cube_dat.nc
Variable Statistics:
--------------------------------------------------
eta:
  Shape: (37, 302, 227)
  Range: [-5.7322, 5.0000]
  Mean:  -1.0276,  Std: 2.4307
water_depth:
  Shape: (37, 302, 227)
  Range: [0.0000, 7.3938]
  Mean:  2.4153,  Std: 1.7003
velocity:
  Shape: (37, 302, 227)
  Range: [0.0000, 1.4721]
  Mean: 

## 3. Additional Variables

By default only `eta`, `water_depth`, `velocity`, `mud_frac`, and `sand_frac` are extracted.
To add more variables, edit `delft3d_dat_converter.py`:

**Step 1 — Read the extra variable inside `extract_variables()`:**
```python
# After the DPS / MUDFRAC reads, add e.g.:
tauksi_raw = r.read_element_all_timesteps('map-sed-series', 'TAUKSI', n_use)
taueta_raw = r.read_element_all_timesteps('map-sed-series', 'TAUETA', n_use)
```

**Step 2 — Compute and store the derived variable:**
```python
bed_shear = np.sqrt(tauksi_raw**2 + taueta_raw**2)
self.data_dict['bed_shear_stress'] = bed_shear
```

**Step 3 — Add attributes** in `_add_variable_attributes()`:
```python
'bed_shear_stress': ('Bed shear stress magnitude', 'N/m^2', 'sqrt(TAUKSI^2 + TAUETA^2)'),
```

### Available optional variables (group → element name):
| Variable | Group | Element | Description |
|----------|-------|---------|-------------|
| Bed shear stress x | `map-sed-series` | `TAUKSI` | Shear stress in x-direction |
| Bed shear stress y | `map-sed-series` | `TAUETA` | Shear stress in y-direction |
| Bed load x | `map-sed-series` | `SBUU` | Bed load transport in x |
| Bed load y | `map-sed-series` | `SBVV` | Bed load transport in y |
| Suspended load x | `map-sed-series` | `SSUU` | Suspended transport in x |
| Suspended load y | `map-sed-series` | `SSVV` | Suspended transport in y |
| Water level | `map-series` | `S1` | Stored separately if needed |